# Mobile Progressive Web App for Card Feeder

This notebook provides the backend API and mobile PWA implementation for phone-mounted card feeder systems.

## Architecture Overview:
- **Frontend**: Progressive Web App with camera access and real-time video streaming
- **Backend**: FastAPI with WebSocket support for real-time processing
- **Processing**: Hybrid approach - motion detection on client, AI processing on server
- **Communication**: WebSocket for real-time bidirectional communication

## Backend API Implementation

In [ ]:
# Install required packages for the backend
!pip install fastapi uvicorn websockets python-multipart aiofiles pillow

In [ ]:
import asyncio
import base64
import io
import json
import logging
import time
from typing import Dict, List, Optional

import cv2
import numpy as np
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import HTMLResponse
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import pinecone
import os
import requests
import csv

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Initialize FastAPI app
app = FastAPI(title="MTG Card Feeder API", version="1.0.0")

# Add CORS middleware for mobile app access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Configure for production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global variables for models and connections
base_model = None
pinecone_index = None
active_connections: List[WebSocket] = []
processing_stats = {
    'cards_processed': 0,
    'total_processing_time': 0,
    'errors': 0,
    'start_time': time.time()
}

In [ ]:
# Initialize AI models and services
async def initialize_services():
    global base_model, pinecone_index
    
    logger.info("Initializing AI models...")
    
    # Load EfficientNetB0 model
    base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
    
    # Warm up the model
    dummy_input = np.random.random((1, 224, 224, 3)).astype(np.float32)
    _ = base_model.predict(dummy_input, verbose=0)
    
    # Initialize Pinecone
    api_key = os.getenv("PINECONE_API_KEY")
    if api_key:
        pinecone_client = pinecone.Pinecone(api_key=api_key)
        pinecone_index = pinecone_client.Index('mtg-cards-index-efficientnet')
        logger.info("Pinecone initialized successfully")
    else:
        logger.warning("Pinecone API key not found")
    
    logger.info("Services initialized successfully")

# Initialize on startup
@app.on_event("startup")
async def startup_event():
    await initialize_services()

In [ ]:
# Image processing functions
def base64_to_image(base64_string: str) -> np.ndarray:
    """Convert base64 string to OpenCV image"""
    try:
        # Remove data URL prefix if present
        if base64_string.startswith('data:image'):
            base64_string = base64_string.split(',')[1]
        
        # Decode base64
        image_data = base64.b64decode(base64_string)
        
        # Convert to PIL Image
        pil_image = Image.open(io.BytesIO(image_data))
        
        # Convert to OpenCV format
        cv_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
        
        return cv_image
    except Exception as e:
        logger.error(f"Error converting base64 to image: {e}")
        return None

def preprocess_image_for_model(image: np.ndarray) -> np.ndarray:
    """Preprocess image for EfficientNet model"""
    try:
        # Resize image to 224x224
        resized = cv2.resize(image, (224, 224))
        
        # Convert to float32 and add batch dimension
        processed = resized.astype(np.float32)
        processed = np.expand_dims(processed, axis=0)
        
        # Apply EfficientNet preprocessing
        processed = preprocess_input(processed)
        
        return processed
    except Exception as e:
        logger.error(f"Error preprocessing image: {e}")
        return None

def generate_embedding(image: np.ndarray) -> Optional[np.ndarray]:
    """Generate embedding from preprocessed image"""
    try:
        processed_image = preprocess_image_for_model(image)
        if processed_image is None:
            return None
        
        embedding = base_model.predict(processed_image, verbose=0)
        return embedding.flatten()
    except Exception as e:
        logger.error(f"Error generating embedding: {e}")
        return None

In [ ]:
# Card identification and database functions
async def identify_card(embedding: np.ndarray) -> Optional[Dict]:
    """Identify card from embedding using Pinecone"""
    try:
        if pinecone_index is None:
            logger.error("Pinecone not initialized")
            return None
        
        # Query Pinecone database
        query_results = pinecone_index.query(
            namespace="mtg_cards",
            vector=embedding.tolist(),
            top_k=3
        )
        
        if not query_results.matches:
            logger.warning("No matches found in database")
            return None
        
        # Get best match
        best_match = query_results.matches[0]
        confidence = best_match.score
        
        if confidence < 0.80:  # Confidence threshold
            logger.warning(f"Low confidence match: {confidence:.3f}")
            return None
        
        # Get card details from Scryfall
        card_id = best_match.id.split('.')[0]
        card_data = await get_card_details(card_id)
        
        if card_data:
            return {
                'card_id': card_id,
                'name': card_data['name'],
                'set_name': card_data.get('set_name', 'Unknown'),
                'rarity': card_data.get('rarity', 'Unknown'),
                'price': card_data.get('prices', {}).get('usd', 'N/A'),
                'confidence': confidence,
                'image_url': card_data.get('image_uris', {}).get('normal', ''),
                'full_data': card_data
            }
        
        return None
    except Exception as e:
        logger.error(f"Error identifying card: {e}")
        return None

async def get_card_details(card_id: str) -> Optional[Dict]:
    """Fetch card details from Scryfall API"""
    try:
        url = f"https://api.scryfall.com/cards/{card_id}"
        response = requests.get(url, timeout=5)
        
        if response.status_code == 200:
            return response.json()
        else:
            logger.error(f"Scryfall API error: {response.status_code}")
            return None
    except Exception as e:
        logger.error(f"Error fetching card details: {e}")
        return None

async def update_catalog(card_data: Dict) -> bool:
    """Update the card catalog with new card"""
    try:
        # This would typically update a database
        # For now, we'll just log the card
        logger.info(f"Adding card to catalog: {card_data['name']}")
        
        # TODO: Implement actual database update
        # - Update PostgreSQL database
        # - Update CSV file
        # - Broadcast to connected clients
        
        return True
    except Exception as e:
        logger.error(f"Error updating catalog: {e}")
        return False

In [ ]:
# WebSocket connection management
class ConnectionManager:
    def __init__(self):
        self.active_connections: List[WebSocket] = []
    
    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)
        logger.info(f"Client connected. Total connections: {len(self.active_connections)}")
    
    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)
        logger.info(f"Client disconnected. Total connections: {len(self.active_connections)}")
    
    async def send_personal_message(self, message: str, websocket: WebSocket):
        await websocket.send_text(message)
    
    async def broadcast(self, message: str):
        for connection in self.active_connections:
            try:
                await connection.send_text(message)
            except:
                # Connection might be closed
                pass

manager = ConnectionManager()

In [ ]:
# WebSocket endpoint for real-time card processing
@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await manager.connect(websocket)
    
    try:
        while True:
            # Receive message from client
            data = await websocket.receive_text()
            message = json.loads(data)
            
            message_type = message.get('type')
            
            if message_type == 'process_frame':
                await handle_frame_processing(websocket, message)
            elif message_type == 'get_stats':
                await send_stats(websocket)
            elif message_type == 'ping':
                await websocket.send_text(json.dumps({'type': 'pong'}))
            else:
                logger.warning(f"Unknown message type: {message_type}")
    
    except WebSocketDisconnect:
        manager.disconnect(websocket)
    except Exception as e:
        logger.error(f"WebSocket error: {e}")
        manager.disconnect(websocket)

async def handle_frame_processing(websocket: WebSocket, message: Dict):
    """Process a frame from the mobile client"""
    start_time = time.time()
    
    try:
        # Extract frame data
        frame_data = message.get('frame')
        if not frame_data:
            await websocket.send_text(json.dumps({
                'type': 'error',
                'message': 'No frame data provided'
            }))
            return
        
        # Convert base64 to image
        image = base64_to_image(frame_data)
        if image is None:
            await websocket.send_text(json.dumps({
                'type': 'error',
                'message': 'Failed to decode image'
            }))
            return
        
        # Generate embedding
        embedding = generate_embedding(image)
        if embedding is None:
            await websocket.send_text(json.dumps({
                'type': 'error',
                'message': 'Failed to generate embedding'
            }))
            return
        
        # Identify card
        card_result = await identify_card(embedding)
        
        processing_time = time.time() - start_time
        
        if card_result:
            # Update catalog
            catalog_updated = await update_catalog(card_result['full_data'])
            
            # Update stats
            processing_stats['cards_processed'] += 1
            processing_stats['total_processing_time'] += processing_time
            
            # Send success response
            await websocket.send_text(json.dumps({
                'type': 'card_identified',
                'card': {
                    'name': card_result['name'],
                    'set_name': card_result['set_name'],
                    'rarity': card_result['rarity'],
                    'price': card_result['price'],
                    'confidence': card_result['confidence'],
                    'image_url': card_result['image_url']
                },
                'processing_time': processing_time,
                'catalog_updated': catalog_updated
            }))
        else:
            # Update error stats
            processing_stats['errors'] += 1
            
            # Send failure response
            await websocket.send_text(json.dumps({
                'type': 'card_not_found',
                'message': 'Card not identified',
                'processing_time': processing_time
            }))
    
    except Exception as e:
        logger.error(f"Error processing frame: {e}")
        await websocket.send_text(json.dumps({
            'type': 'error',
            'message': f'Processing error: {str(e)}'
        }))

async def send_stats(websocket: WebSocket):
    """Send processing statistics to client"""
    runtime = time.time() - processing_stats['start_time']
    cards_processed = processing_stats['cards_processed']
    avg_processing_time = (
        processing_stats['total_processing_time'] / cards_processed
        if cards_processed > 0 else 0
    )
    
    await websocket.send_text(json.dumps({
        'type': 'stats',
        'data': {
            'cards_processed': cards_processed,
            'errors': processing_stats['errors'],
            'avg_processing_time': avg_processing_time,
            'runtime': runtime,
            'throughput': cards_processed / (runtime / 60) if runtime > 0 else 0
        }
    }))

In [ ]:
# REST API endpoints
@app.get("/")
async def root():
    return {"message": "MTG Card Feeder API", "version": "1.0.0"}

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "model_loaded": base_model is not None,
        "pinecone_connected": pinecone_index is not None,
        "active_connections": len(manager.active_connections)
    }

@app.get("/stats")
async def get_stats():
    runtime = time.time() - processing_stats['start_time']
    cards_processed = processing_stats['cards_processed']
    
    return {
        "cards_processed": cards_processed,
        "errors": processing_stats['errors'],
        "avg_processing_time": (
            processing_stats['total_processing_time'] / cards_processed
            if cards_processed > 0 else 0
        ),
        "runtime_seconds": runtime,
        "throughput_per_minute": cards_processed / (runtime / 60) if runtime > 0 else 0,
        "active_connections": len(manager.active_connections)
    }

In [ ]:
# Save the backend server as a Python file
backend_code = '''
import asyncio
import base64
import io
import json
import logging
import time
from typing import Dict, List, Optional

import cv2
import numpy as np
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import pinecone
import os
import requests
import uvicorn

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="MTG Card Feeder API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ... (Include all the functions from above) ...

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('mtg_card_feeder_api.py', 'w') as f:
    f.write(backend_code)

print("Backend API saved to mtg_card_feeder_api.py")
print("To run the server: python mtg_card_feeder_api.py")

## Progressive Web App (PWA) Implementation

In [ ]:
# Create the PWA HTML file
pwa_html = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>MTG Card Feeder</title>
    <link rel="manifest" href="manifest.json">
    <meta name="theme-color" content="#1a1a1a">
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }
        
        body {
            font-family: Arial, sans-serif;
            background: #1a1a1a;
            color: white;
            overflow: hidden;
        }
        
        .container {
            display: flex;
            flex-direction: column;
            height: 100vh;
        }
        
        .header {
            background: #2a2a2a;
            padding: 10px;
            text-align: center;
            font-size: 18px;
            font-weight: bold;
        }
        
        .camera-container {
            flex: 1;
            position: relative;
            background: black;
        }
        
        #video {
            width: 100%;
            height: 100%;
            object-fit: cover;
        }
        
        .overlay {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            pointer-events: none;
        }
        
        .card-outline {
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            width: 200px;
            height: 280px;
            border: 2px solid #00ff00;
            border-radius: 10px;
            opacity: 0.7;
        }
        
        .status-bar {
            background: #2a2a2a;
            padding: 15px;
            min-height: 100px;
        }
        
        .status-text {
            font-size: 14px;
            margin-bottom: 5px;
        }
        
        .card-result {
            background: #333;
            border-radius: 5px;
            padding: 10px;
            margin-top: 10px;
            display: none;
        }
        
        .card-name {
            font-size: 16px;
            font-weight: bold;
            color: #00ff00;
        }
        
        .card-details {
            font-size: 12px;
            color: #ccc;
            margin-top: 5px;
        }
        
        .controls {
            display: flex;
            gap: 10px;
            margin-top: 10px;
        }
        
        button {
            flex: 1;
            padding: 10px;
            border: none;
            border-radius: 5px;
            background: #007bff;
            color: white;
            font-size: 14px;
            cursor: pointer;
        }
        
        button:hover {
            background: #0056b3;
        }
        
        button:disabled {
            background: #666;
            cursor: not-allowed;
        }
        
        .error {
            color: #ff6b6b;
            font-size: 12px;
        }
        
        .success {
            color: #00ff00;
            font-size: 12px;
        }
        
        .stats {
            display: flex;
            justify-content: space-between;
            font-size: 11px;
            color: #999;
            margin-top: 5px;
        }
        
        .processing {
            color: #ffa500;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            MTG Card Feeder
        </div>
        
        <div class="camera-container">
            <video id="video" playsinline autoplay muted></video>
            <div class="overlay">
                <div class="card-outline" id="cardOutline"></div>
            </div>
        </div>
        
        <div class="status-bar">
            <div class="status-text" id="statusText">Connecting...</div>
            <div class="stats">
                <span>Cards: <span id="cardCount">0</span></span>
                <span>Errors: <span id="errorCount">0</span></span>
                <span>Avg Time: <span id="avgTime">0</span>s</span>
            </div>
            
            <div class="card-result" id="cardResult">
                <div class="card-name" id="cardName"></div>
                <div class="card-details" id="cardDetails"></div>
            </div>
            
            <div class="controls">
                <button id="startBtn" onclick="startProcessing()">Start</button>
                <button id="stopBtn" onclick="stopProcessing()" disabled>Stop</button>
                <button id="statsBtn" onclick="showStats()">Stats</button>
            </div>
        </div>
    </div>

    <script>
        let video, canvas, ctx, ws;
        let isProcessing = false;
        let processingInterval = null;
        let stats = { cards: 0, errors: 0, avgTime: 0 };
        
        // WebSocket connection
        const WS_URL = 'ws://localhost:8000/ws';
        
        // Initialize camera and WebSocket
        async function init() {
            video = document.getElementById('video');
            canvas = document.createElement('canvas');
            ctx = canvas.getContext('2d');
            
            try {
                // Request camera access
                const stream = await navigator.mediaDevices.getUserMedia({
                    video: {
                        facingMode: 'environment', // Use back camera
                        width: { ideal: 1920 },
                        height: { ideal: 1080 }
                    }
                });
                
                video.srcObject = stream;
                updateStatus('Camera connected', 'success');
                
                // Connect to WebSocket
                connectWebSocket();
                
            } catch (error) {
                updateStatus('Camera access denied: ' + error.message, 'error');
            }
        }
        
        function connectWebSocket() {
            ws = new WebSocket(WS_URL);
            
            ws.onopen = function() {
                updateStatus('Connected to server', 'success');
                document.getElementById('startBtn').disabled = false;
            };
            
            ws.onmessage = function(event) {
                const data = JSON.parse(event.data);
                handleWebSocketMessage(data);
            };
            
            ws.onclose = function() {
                updateStatus('Disconnected from server', 'error');
                document.getElementById('startBtn').disabled = true;
                document.getElementById('stopBtn').disabled = true;
            };
            
            ws.onerror = function(error) {
                updateStatus('WebSocket error: ' + error.message, 'error');
            };
        }
        
        function handleWebSocketMessage(data) {
            switch(data.type) {
                case 'card_identified':
                    displayCardResult(data.card);
                    stats.cards++;
                    updateStats();
                    updateStatus('Card identified: ' + data.card.name, 'success');
                    break;
                    
                case 'card_not_found':
                    updateStatus('Card not found', 'error');
                    stats.errors++;
                    updateStats();
                    break;
                    
                case 'error':
                    updateStatus('Error: ' + data.message, 'error');
                    stats.errors++;
                    updateStats();
                    break;
                    
                case 'stats':
                    stats = {
                        cards: data.data.cards_processed,
                        errors: data.data.errors,
                        avgTime: data.data.avg_processing_time
                    };
                    updateStats();
                    break;
            }
        }
        
        function captureFrame() {
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0);
            
            // Convert to base64
            const dataURL = canvas.toDataURL('image/jpeg', 0.8);
            return dataURL;
        }
        
        function startProcessing() {
            if (!ws || ws.readyState !== WebSocket.OPEN) {
                updateStatus('Not connected to server', 'error');
                return;
            }
            
            isProcessing = true;
            document.getElementById('startBtn').disabled = true;
            document.getElementById('stopBtn').disabled = false;
            
            updateStatus('Processing started', 'success');
            
            // Process frames every 2 seconds
            processingInterval = setInterval(() => {
                if (isProcessing) {
                    const frameData = captureFrame();
                    
                    ws.send(JSON.stringify({
                        type: 'process_frame',
                        frame: frameData
                    }));
                    
                    updateStatus('Processing frame...', 'processing');
                }
            }, 2000);
        }
        
        function stopProcessing() {
            isProcessing = false;
            
            if (processingInterval) {
                clearInterval(processingInterval);
                processingInterval = null;
            }
            
            document.getElementById('startBtn').disabled = false;
            document.getElementById('stopBtn').disabled = true;
            
            updateStatus('Processing stopped', 'success');
        }
        
        function displayCardResult(card) {
            const resultDiv = document.getElementById('cardResult');
            const nameDiv = document.getElementById('cardName');
            const detailsDiv = document.getElementById('cardDetails');
            
            nameDiv.textContent = card.name;
            detailsDiv.textContent = `${card.set_name} • ${card.rarity} • $${card.price} • ${(card.confidence * 100).toFixed(1)}%`;
            
            resultDiv.style.display = 'block';
            
            // Hide after 5 seconds
            setTimeout(() => {
                resultDiv.style.display = 'none';
            }, 5000);
        }
        
        function updateStatus(message, type) {
            const statusText = document.getElementById('statusText');
            statusText.textContent = message;
            statusText.className = 'status-text ' + (type || '');
        }
        
        function updateStats() {
            document.getElementById('cardCount').textContent = stats.cards;
            document.getElementById('errorCount').textContent = stats.errors;
            document.getElementById('avgTime').textContent = stats.avgTime.toFixed(2);
        }
        
        function showStats() {
            if (ws && ws.readyState === WebSocket.OPEN) {
                ws.send(JSON.stringify({ type: 'get_stats' }));
            }
        }
        
        // Initialize when page loads
        window.addEventListener('load', init);
        
        // Register service worker for PWA
        if ('serviceWorker' in navigator) {
            navigator.serviceWorker.register('sw.js');
        }
    </script>
</body>
</html>
'''

with open('index.html', 'w') as f:
    f.write(pwa_html)

print("PWA HTML file created: index.html")

In [ ]:
# Create PWA manifest
manifest = '''
{
  "name": "MTG Card Feeder",
  "short_name": "MTG Feeder",
  "description": "Automated Magic: The Gathering card identification and cataloging",
  "start_url": "/",
  "display": "standalone",
  "background_color": "#1a1a1a",
  "theme_color": "#1a1a1a",
  "icons": [
    {
      "src": "icon-192.png",
      "sizes": "192x192",
      "type": "image/png"
    },
    {
      "src": "icon-512.png",
      "sizes": "512x512",
      "type": "image/png"
    }
  ],
  "orientation": "portrait",
  "categories": ["games", "utilities"]
}
'''

with open('manifest.json', 'w') as f:
    f.write(manifest)

print("PWA manifest created: manifest.json")

In [ ]:
# Create service worker for PWA
service_worker = '''
const CACHE_NAME = 'mtg-card-feeder-v1';
const urlsToCache = [
  '/',
  '/index.html',
  '/manifest.json'
];

self.addEventListener('install', function(event) {
  event.waitUntil(
    caches.open(CACHE_NAME)
      .then(function(cache) {
        return cache.addAll(urlsToCache);
      })
  );
});

self.addEventListener('fetch', function(event) {
  event.respondWith(
    caches.match(event.request)
      .then(function(response) {
        // Return cached version or fetch from network
        return response || fetch(event.request);
      }
    )
  );
});
'''

with open('sw.js', 'w') as f:
    f.write(service_worker)

print("Service worker created: sw.js")

## Deployment Instructions

In [ ]:
# Create deployment instructions
deployment_instructions = '''
# MTG Card Feeder Mobile PWA Deployment Guide

## Backend Deployment

### 1. Install Dependencies
```bash
pip install fastapi uvicorn websockets python-multipart aiofiles
pip install tensorflow opencv-python pillow pinecone-client
```

### 2. Set Environment Variables
```bash
export PINECONE_API_KEY=your_pinecone_api_key
```

### 3. Run Backend Server
```bash
python mtg_card_feeder_api.py
```

## Frontend Deployment

### 1. Serve Static Files
```bash
# Using Python's built-in server
python -m http.server 8080

# Or using Node.js serve
npm install -g serve
serve -s . -p 8080
```

### 2. Access the App
- Open browser to: http://localhost:8080
- On mobile: Connect to same WiFi network and use your computer's IP
- Example: http://192.168.1.100:8080

## Production Deployment

### Backend (Cloud)
1. Deploy to cloud service (AWS, Google Cloud, etc.)
2. Use Docker for containerization
3. Set up HTTPS for secure WebSocket connections
4. Configure load balancing for multiple instances

### Frontend (CDN)
1. Host static files on CDN (Cloudflare, AWS S3, etc.)
2. Enable HTTPS
3. Configure PWA installation prompts
4. Set up offline caching strategies

## Mobile Usage

### 1. Install PWA
- Open in mobile browser
- Tap "Add to Home Screen" when prompted
- Launch from home screen for app-like experience

### 2. Mount Phone to Card Feeder
- Ensure stable positioning
- Good lighting conditions
- Cards feed through camera view

### 3. Start Processing
- Tap "Start" button
- Feed cards through at 2-second intervals
- Monitor results in real-time

## Performance Optimization

### Backend
- Use GPU acceleration if available
- Implement connection pooling
- Add Redis caching for frequent queries
- Monitor memory usage and optimize batch sizes

### Frontend
- Implement adaptive quality based on network speed
- Add offline mode for poor connections
- Optimize image compression before upload
- Use WebWorkers for background processing

## Troubleshooting

### Common Issues
1. **Camera not working**: Check permissions and use HTTPS
2. **WebSocket connection failed**: Verify backend is running
3. **Slow processing**: Check network connection and server resources
4. **Poor card detection**: Improve lighting and card positioning
'''

with open('deployment_guide.md', 'w') as f:
    f.write(deployment_instructions)

print("Deployment guide created: deployment_guide.md")

## Testing the System

In [ ]:
# Test the WebSocket connection locally
import asyncio
import websockets
import json
import base64

async def test_websocket():
    uri = "ws://localhost:8000/ws"
    
    try:
        async with websockets.connect(uri) as websocket:
            print("Connected to WebSocket server")
            
            # Send ping
            await websocket.send(json.dumps({"type": "ping"}))
            response = await websocket.recv()
            print(f"Ping response: {response}")
            
            # Request stats
            await websocket.send(json.dumps({"type": "get_stats"}))
            response = await websocket.recv()
            print(f"Stats response: {response}")
            
    except Exception as e:
        print(f"WebSocket test failed: {e}")
        print("Make sure the backend server is running on localhost:8000")

# Uncomment to test WebSocket connection
# asyncio.run(test_websocket())